# Experiment 00 — Build the MIMIC-CXR-JPG subsample manifest

Produces `manifests/mimic_subsample_ids.parquet`, the reproducible manifest of the exact
(dicom_id, subject_id, study_id, split) rows used by every MIMIC experiment in the pipeline.

**Design.**
- Frontal views only (AP/PA), matching NIH / Emory protocol.
- Official MIMIC-CXR train/validate/test split is preserved verbatim. Patient-disjointness across splits is guaranteed by PhysioNet.
- Evaluation (test + validate) uses the **full** official sets — no subsampling.
- Probe-training is stratified-subsampled within the official train split:
  - Stratification key: 3-bit label pattern (Cardiomegaly, Edema, Lung Lesion; CheXpert U-Zero convention).
  - Target $n$ = `TARGET_N_TRAIN` (default 50 000).
  - Seed fixed at 42.
- Output columns kept at image (`dicom_id`) level so downstream code filters with a single `isin()` call.


In [ ]:
# === Papermill parameters (override via `papermill -p NAME VALUE`) ===
TARGET_N_TRAIN = 50_000
SEED = 42
MIMIC_ROOT = "/data0/MIMIC-CXR"
OUT_PATH = "/home/saptpurk/embeddings-noise-eliminators/manifests/mimic_subsample_ids.parquet"


In [ ]:
from __future__ import annotations
from pathlib import Path
import numpy as np
import pandas as pd

MIMIC_ROOT = Path(MIMIC_ROOT)
OUT_PATH = Path(OUT_PATH)
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)


## 1. Load MIMIC metadata, labels, and split files


In [ ]:
meta = pd.read_csv(MIMIC_ROOT / "mimic-cxr-2.0.0-metadata.csv.gz")
lab  = pd.read_csv(MIMIC_ROOT / "mimic-cxr-2.0.0-chexpert.csv.gz")
spl  = pd.read_csv(MIMIC_ROOT / "mimic-cxr-2.0.0-split.csv.gz")
print(f"raw metadata rows: {len(meta):,}")
print(f"raw label    rows: {len(lab):,}")
print(f"raw split    rows: {len(spl):,}")


## 2. Filter frontal views; merge metadata + labels + splits


In [ ]:
meta = meta[meta["ViewPosition"].isin(["AP", "PA"])].copy()
df = meta.merge(lab, on=["subject_id", "study_id"], how="inner")
df = df.merge(
    spl[["subject_id", "study_id", "dicom_id", "split"]],
    on=["subject_id", "study_id", "dicom_id"], how="left",
)
df = df[df["split"].isin(["train", "validate", "test"])].copy()

# CheXpert U-Zero: NaN -> 0; uncertain (-1) treated as negative; positive (1) kept positive
for d_col in ["Cardiomegaly", "Edema", "Lung Lesion"]:
    df[d_col] = (df[d_col].fillna(0).astype(float) > 0.5).astype(int)

df["strata"] = (df["Cardiomegaly"].astype(str)
                + df["Edema"].astype(str)
                + df["Lung Lesion"].astype(str))

print(f"frontal+labelled+split rows: {len(df):,}")
print(df['split'].value_counts().to_string())


## 3. Stratified subsample of the train split (preserve validate/test fully)


In [ ]:
keep_test = df[df["split"] == "test"].copy()
keep_val  = df[df["split"] == "validate"].copy()
train     = df[df["split"] == "train"].copy()

frac = min(1.0, TARGET_N_TRAIN / len(train))

def _pick(g):
    n = max(1, int(round(len(g) * frac)))
    return g.sample(n=min(n, len(g)), random_state=SEED)

train_sub = (train.groupby("strata", group_keys=False)
                  .apply(_pick)
                  .reset_index(drop=True))

out = pd.concat([train_sub, keep_val, keep_test], ignore_index=True)
out = out.assign(subsampled=True)
out = out[["dicom_id", "subject_id", "study_id", "split", "strata",
           "Cardiomegaly", "Edema", "Lung Lesion", "subsampled"]]

out.to_parquet(OUT_PATH, index=False)
print(f"Wrote {len(out):,} rows to {OUT_PATH}")


## 4. Sanity diagnostics: per-split counts and label-prevalence drift


In [ ]:
print("Per-split counts (frontal, post-subsample):")
print(out.groupby('split').size().to_string())

print('\nPer-split label prevalence:')
for s in ['train', 'validate', 'test']:
    sub = out[out['split'] == s]
    row = {d: f'{sub[d].mean():.3f}' for d in ('Cardiomegaly', 'Edema', 'Lung Lesion')}
    print(f'  {s:8s}  n={len(sub):7,d}  {row}')

full_train = df[df['split'] == 'train']
print('\nFull-train vs subsample prevalence drift (pp):')
for d in ('Cardiomegaly', 'Edema', 'Lung Lesion'):
    drift = (train_sub[d].mean() - full_train[d].mean()) * 100
    print(f'  {d:15s}  full={full_train[d].mean():.3f}  '
          f'sub={train_sub[d].mean():.3f}  delta={drift:+.2f}pp')
